#TRACING

In [2]:
import os
import time
from typing import List
from dotenv import load_dotenv
load_dotenv() 
PERPLEXITY_KEY = os.getenv("PERPLEXITY_API_KEY")
LANGSMITH_KEY = os.getenv("LANGSMITH_API_KEY")


if not PERPLEXITY_KEY:
    raise RuntimeError("PERPLEXITY_API_KEY missing in environment (.env).")
if not LANGSMITH_KEY:
    print("Warning: LANGSMITH_API_KEY not set. Traces will not upload to LangSmith.")
from langsmith import traceable  
from langchain_perplexity import ChatPerplexity
from langchain_core.messages import SystemMessage, HumanMessage  
model = ChatPerplexity(api_key=PERPLEXITY_KEY, model="sonar-pro")  
def get_vector_db_retriever():
    """
    Replace this with your actual retriever. For example, a Chroma/FAISS retriever wrapped with LangChain.
    The retriever must return an iterable of documents with `page_content` attribute.
    """
    class Doc:
        def __init__(self, text):
            self.page_content = text
    return lambda q: [Doc("OHLC graphs show open-high-low-close prices and are used in candlestick analysis.")]
retriever = get_vector_db_retriever()
RAG_SYSTEM_PROMPT = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question. 
If you don't know the answer, say that you don't know. Use a maximum of three sentences and be concise."""
@traceable(metadata={"component": "retriever"})
def retrieve_documents(question: str):
    """Call the retriever (vector DB) and return documents."""
    docs = retriever(question)  
    return docs
@traceable(metadata={"component": "prompt_builder"})
def generate_response_prompt(question: str, documents):
    """Format the retrieved documents into a prompt/messages for the model."""
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    messages = [
        SystemMessage(RAG_SYSTEM_PROMPT),
        HumanMessage(f"Context: {formatted_docs}\n\nQuestion: {question}")
    ]
    return messages

@traceable(metadata={"component": "model_call", "model_provider": "perplexity", "model_name": "sonar"})
def call_perplexity(messages: List[dict], temperature: float = 0.0):
    """Call Perplexity chat model and return the raw response object."""
    return model.invoke(messages)

@traceable(metadata={"component": "pipeline"})
def langsmith_rag(question: str, langsmith_extra: dict = None):
    """
    Top-level function that will be traced as the root run.
    You can optionally pass langsmith_extra = {"metadata": {...}, "tags": [...]} to add runtime metadata.
    """
    documents = retrieve_documents(question)
    messages = generate_response_prompt(question, documents)
    response = call_perplexity(messages)
    content = getattr(response, "content", None) or (
        response.choices[0].message.content if hasattr(response, "choices") else str(response)
    )
    return content

file_url = "file:///mnt/data/570b7b58-f96b-4394-84e6-73c7046db8e4.png"

q = "How can I interpret OHLC candlestick wicks for intraday trading?"
extra = {"metadata": {"module": "module_1", "lesson": "lesson_1", "file_reference": file_url}}
answer = langsmith_rag(q, langsmith_extra=extra)
print("Model answer:\n", answer)


Model answer:
 For intraday trading, **OHLC candlestick wicks** indicate how far price moved beyond the open-close range within each candle’s timeframe, revealing volatility, potential reversals, and market momentum[1][5][9]. 

- **Long upper wicks** suggest buyers pushed price up but sellers forced it back down, often indicating selling pressure or potential resistance, and can signal a reversal from an up-move if seen after a rally[5][9].
- **Long lower wicks** show sellers drove price down but buyers pushed it back up, indicating buying pressure or support, and may signal a reversal from a down-move if seen after a decline[5][9].
- **Short or absent wicks** typically indicate strong directional moves with little opposition—useful for identifying momentum continuation[1][3].

Intraday traders often watch wick length in real time to spot areas of rejection and gauge the strength of price movements, helping to identify trade entries and exits within the day[1][9][12].


#TOOL CALLING

In [13]:
import os, json
from dotenv import load_dotenv
from langsmith import traceable
from langchain_perplexity import ChatPerplexity
load_dotenv()
PKEY = os.getenv("PERPLEXITY_API_KEY")
model = ChatPerplexity(api_key=PKEY, model="sonar")
@traceable(run_type="tool")
def find_distance(a: str, b: str):
    distances = {("mumbai","delhi"): 1400, ("new york","boston"): 350}
    key = (a.lower().strip(), b.lower().strip())
    km = distances.get(key, 999)  # fallback value
    return {"from": a, "to": b, "distance_km": km}
@traceable(run_type="llm")
def call_llm(messages):
    r = model.invoke(messages)
    return getattr(r, "content", str(r))
@traceable(run_type="chain")
def distance_finding(question):
    msgs = [
      {"role":"system", "content": "If you need a tool, reply with JSON: {\"tool\": {\"name\":\"find_distance\",\"args\":{...}}}"},
      {"role":"user", "content": question + f"\nReference: {file_url}"}
    ]
    first = call_llm(msgs).strip()
    tool = None
    try:
        s, e = first.find("{"), first.rfind("}")
        if s!=-1 and e> s:
            parsed = json.loads(first[s:e+1])
            tool = parsed.get("tool")
    except Exception:
        tool = None
    if tool and tool.get("name")=="find_distance":
        a = tool["args"].get("from") or tool["args"].get("a") or ""
        b = tool["args"].get("to")   or tool["args"].get("b") or ""
        result = find_distance(a, b)
        msgs.append({"role":"assistant","content": first})
        msgs.append({"role":"tool","content": json.dumps(result)})
        return call_llm(msgs)
    return first
print(distance_finding("What is the distance between Mumbai and Delhi (use km)?"))


The distance between Mumbai and Delhi is approximately **1148 to 1153 kilometers** by air (straight line distance). The flight distance commonly cited is around **1148 km to 1153 km** (about 713 to 716 miles)[1][2][3][6].

Additional details:
- The **driving distance** is longer, ranging from approximately **1360 km to 1447 km** depending on the route[1][2][5][6].
- Flight times between Mumbai and Delhi are roughly 3 to 3.5 hours, reflecting the air distance of around 1148 km[7][9].
- Some sources give slightly varying distances due to different calculation methods (e.g., Vincenty's formula, Haversine formula), but all agree on roughly 1140-1150 km air distance[5][7].

Therefore, for practical purposes, the **distance between Mumbai and Delhi is about 1150 kilometers by air**.
